In [ ]:

from pathlib import Path
import xarray as xr
from scipy.stats import binned_statistic_2d
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
  # From other external Python packages:
from datetime import datetime, timedelta 
import os
import numpy as np
import netCDF4 as nc
import matplotlib as mpl
from processor import process_granule_directory, create_orthographic_plot, create_global_plot

In [ ]:
data_folder = "/data/wdu36"
month_range = range(1, 13)
for channel_idx in [12, 24, 30, 32]:
    granule_dirs = [
    # "/data/wdu36/sat1_01_26",
    # "/data/wdu36/sat2_01_26",
    ]
    for month_idx in month_range:
        granule_dirs.append(f"{data_folder}/sat1_{month_idx:02d}_25")
        granule_dirs.append(f"{data_folder}/sat2_{month_idx:02d}_25")
    all_missed = []
    for granule_dir in granule_dirs:
        sat_num = granule_dir.split('_')[0][-1]
        mmyy = granule_dir[-5:-3]+granule_dir[-2:]
        print(mmyy)
        try:
            tb_all, lat_all, lon_all, wavelength = process_granule_directory([granule_dir], channel_index = channel_idx)
        except Exception as e:
            print(f"Error processing directory {granule_dir}: {e}")
            all_missed.append(granule_dir)
            continue
            
        # Mask out unrealistic brightness temperature values
        # Remove values below 50 K, above 500 K, and fill values
        realistic_mask = (tb_all >= 150) & (tb_all <= 350)

        # Apply mask to keep arrays in sync
        tb_all = tb_all[realistic_mask]
        lat_all = lat_all[realistic_mask]
        lon_all = lon_all[realistic_mask]

        lat_edges = np.arange(-90,91,1)
        lon_edges = np.arange(-180,181,1)

        lat_mids = lat_edges[:-1] + 0.5
        lon_mids = lon_edges[:-1] + 0.5
        mean_tb, _, _, _ = binned_statistic_2d(
            lat_all, lon_all, tb_all,
            statistic='mean',
            bins=[lat_edges, lon_edges]
        )

        count_tb, _, _, _ = binned_statistic_2d(
            lat_all, lon_all, tb_all,
            statistic='count',
            bins=[lat_edges, lon_edges]
        )

        std_tb, _, _, _ = binned_statistic_2d(
            lat_all, lon_all, tb_all,
            statistic='std',
            bins=[lat_edges, lon_edges]
        )

        max_tb, _, _, _ = binned_statistic_2d(
            lat_all, lon_all, tb_all,
            statistic='max',
            bins=[lat_edges, lon_edges]
        )

        min_tb, _, _, _ = binned_statistic_2d(
            lat_all, lon_all, tb_all,
            statistic='min',
            bins=[lat_edges, lon_edges]
        )
        lon2d, lat2d = np.meshgrid(lon_mids, lat_mids)
        threshold=1000
        count_mask = (count_tb > threshold)
        ds = xr.Dataset(
            {
                'mean_tb': (['lat', 'lon'], mean_tb, {'short_name': 'Mean BT', 'long_name': 'Mean Brightness Temperature', 'units': 'K'}),
                'count_tb': (['lat', 'lon'], count_tb, {'short_name': 'Count TB', 'long_name': 'Count of  Observations', 'units': 'count'}),
                'std_tb': (['lat', 'lon'], std_tb, {'short_name': 'Std BT', 'long_name': 'Standard Deviation of Brightness Temperature', 'units': 'K'}),
                'max_tb': (['lat', 'lon'], max_tb, {'short_name': 'Max BT', 'long_name': 'Maximum Brightness Temperature', 'units': 'K'}),
                'min_tb': (['lat', 'lon'], min_tb, {'short_name': 'Min BT', 'long_name': 'Minimum Brightness Temperature', 'units': 'K'}),
                'count_mask': (['lat', 'lon'], count_mask.astype(float), {'short_name': 'Count Mask', 'long_name': 'Count Mask', 'units': 'count'})
            },
            coords={
                'lat': (['lat'], lat_mids, {'units': 'degrees_north'}), 
                'lon': (['lon'], lon_mids, {'units': 'degrees_east'})
            },
            attrs={
                'time': mmyy,
                'channel': channel_idx,
                'wavelength': wavelength,
                'wavelength_units': 'µm',
                'description': f'SAT{sat_num} brightness temperature statistics for channel {channel_idx} for {mmyy}',
            }
        )
        ds.to_netcdf(f'{data_folder}/sat{sat_num}_{mmyy}_ch{channel_idx}.nc')

In [ ]:
ds_test = []
errors1 = []
attr = 'webp'
data_folder = "/data/wdu36"
folder = f'/home/wdu36/new_plots/season_plots/'
for channel in [12, 24, 30, 32]:
    for sattlite in ['sat1', 'sat2']:
        for year in ['2024', '2025', '2026']:
            for season in ['SON','DJF']:
                year_short = year[-2:]
                try:
                    ds_test = []
                    if season == 'DJF':
                        names = [f'{sattlite}_12{year_short}', f'{sattlite}_01{int(year_short)+1}', f'{sattlite}_02{int(year_short)+1}']
                    elif season == 'MAM':
                        names = [f'{sattlite}_03{year_short}', f'{sattlite}_04{year_short}', f'{sattlite}_05{year_short}']
                    elif season == 'JJA':
                        names = [f'{sattlite}_06{year_short}', f'{sattlite}_07{year_short}', f'{sattlite}_08{year_short}']
                    elif season == 'SON':
                        names = [f'{sattlite}_09{year_short}', f'{sattlite}_10{year_short}', f'{sattlite}_11{year_short}']

                    wavelength = 0

                    for name in names:
                        ds = xr.load_dataset(f'{data_folder}/{name}_ch{channel}.nc')
                        ds = ds.assign_coords(time=datetime.strptime(name.split('_')[1], '%m%y'))
                        ds_test.append(ds)
                        wavelength = ds.wavelength
                    ds_test = xr.concat(ds_test, dim='time')
                    ds_test = ds_test.assign_attrs({'wavelength': wavelength})
                    ds_test = ds_test.mean(dim='time')

                    if f'sp_ch{channel}' not in os.listdir(f'{folder}'):
                        os.mkdir(f'{folder}sp_ch{channel}')
                    if f'gb_ch{channel}' not in os.listdir(f'{folder}'):
                        os.mkdir(f'{folder}gb_ch{channel}')
                    if f'np_ch{channel}' not in os.listdir(f'{folder}'):
                        os.mkdir(f'{folder}np_ch{channel}')
                    for var in ['mean_tb', 'std_tb', 'min_tb', 'max_tb']:
                        # ds_test[var] = ds_test[var].where(count_mask > 0)
                        var_name = var.split('_')[0]
                        #sat1_1_2024_np_ch12_mean.png
                        file = f"{sattlite}_{season}_{year}_np_ch{channel}_{var_name}.{attr}"
                        data = ds_test[var].sel(lat = slice(60, 85))
                        vmin = np.floor(np.nanpercentile(data.values, 3) / 10) * 10
                        vmax = np.ceil(np.nanpercentile(data.values, 97) / 10) * 10
                        create_orthographic_plot(ds_test[var], statistic_name=var_name, pole='north', 
                            wavelength=wavelength, output_path=f'{folder}np_ch{channel}/{file}', 
                            count_mask=ds_test.count_mask, satellite=sattlite.capitalize(), season=season, vmin=vmin, vmax=vmax)
                        
                        file = f"{sattlite}_{season}_{year}_sp_ch{channel}_{var_name}.{attr}"
                        data = ds_test[var].sel(lat = slice(-85, -60))#.sel(lat = slice(60, 85))
                        vmin = np.floor(np.nanpercentile(data.values, 3) / 10) * 10
                        vmax = np.ceil(np.nanpercentile(data.values, 97) / 10) * 10
                        create_orthographic_plot(ds_test[var], statistic_name=var_name, pole='south', 
                            wavelength=wavelength, output_path=f'{folder}sp_ch{channel}/{file}', 
                            count_mask=ds_test.count_mask, satellite=sattlite.capitalize(), season=season, vmin=vmin, vmax=vmax)
                        
                        file = f"{sattlite}_{season}_{year}_gb_ch{channel}_{var_name}.{attr}"
                        data = ds_test[var]
                        vmin = np.floor(np.nanpercentile(data.values, 3) / 10) * 10
                        vmax = np.ceil(np.nanpercentile(data.values, 97) / 10) * 10
                        create_global_plot(ds_test[var], statistic_name=var_name, 
                            wavelength=wavelength, output_path=f'{folder}gb_ch{channel}/{file}', 
                            count_mask=ds_test.count_mask, satellite=sattlite.capitalize(), season=season, vmin=vmin, vmax=vmax)
                except:
                    errors1.append(f"Error processing {sattlite} {season} ch{channel}")
                    

In [ ]:
start = datetime.strptime("0824", "%m%y")
end = datetime.strptime("0226", "%m%y")

months = []
current = start

while current <= end:
    months.append(current.strftime("%m%y"))
    if current.month == 12:
        current = current.replace(year=current.year + 1, month=1)
    else:
        current = current.replace(month=current.month + 1)

months

In [ ]:
ds_test = []
errors = []
attr = 'webp'
data_folder = "/data/wdu36"
folder = f'/home/wdu36/new_plots/month_plots/'
for channel in [12, 24, 30, 32]:
    for sattlite in ['sat1', 'sat2']:
        # month = 'SON'
        for month in months:
            try:
                ds_test = []
                name = f'{sattlite}_{month}'
                wavelength = 0

                ds = xr.load_dataset(f'{data_folder}/{name}_ch{channel}.nc')
                ds_test=ds
                wavelength = ds.wavelength
                ds_test = ds_test.assign_attrs({'wavelength': wavelength})

                if f'sp_ch{channel}' not in os.listdir(f'{folder}'):
                    os.mkdir(f'{folder}sp_ch{channel}')
                if f'gb_ch{channel}' not in os.listdir(f'{folder}'):
                    os.mkdir(f'{folder}gb_ch{channel}')
                if f'np_ch{channel}' not in os.listdir(f'{folder}'):
                    os.mkdir(f'{folder}np_ch{channel}')
                for var in ['mean_tb', 'std_tb', 'min_tb', 'max_tb']:
                    # ds_test[var] = ds_test[var].where(count_mask > 0)
                    var_name = var.split('_')[0]
                    dt = datetime.strptime(month, "%m%y")
                    file = f"{sattlite}_{dt.month}_{dt.year}_np_ch{channel}_{var_name}.{attr}"
                    data = ds_test[var].sel(lat = slice(60, 85))
                    vmin = np.floor(np.nanpercentile(data.values, 3) / 10) * 10
                    vmax = np.ceil(np.nanpercentile(data.values, 97) / 10) * 10
                    month_str = datetime.strptime(month, "%m%y").strftime("%b %Y")
                    create_orthographic_plot(ds_test[var], statistic_name=var_name, pole='north', 
                        wavelength=wavelength, output_path=f'{folder}np_ch{channel}/{file}', 
                        count_mask=ds_test.count_mask, satellite=sattlite.capitalize(), season=month_str, vmin=vmin, vmax=vmax)
                    
                    file = f"{sattlite}_{dt.month}_{dt.year}_sp_ch{channel}_{var_name}.{attr}"
                    data = ds_test[var].sel(lat = slice(-85, -60))#.sel(lat = slice(60, 85))
                    vmin = np.floor(np.nanpercentile(data.values, 3) / 10) * 10
                    vmax = np.ceil(np.nanpercentile(data.values, 97) / 10) * 10
                    month_str = datetime.strptime(month, "%m%y").strftime("%b %Y")
                    create_orthographic_plot(ds_test[var], statistic_name=var_name, pole='south', 
                        wavelength=wavelength, output_path=f'{folder}sp_ch{channel}/{file}', 
                        count_mask=ds_test.count_mask, satellite=sattlite.capitalize(), season=month_str, vmin=vmin, vmax=vmax)
                    
                    file = f"{sattlite}_{dt.month}_{dt.year}_gb_ch{channel}_{var_name}.{attr}"
                    data = ds_test[var]
                    vmin = np.floor(np.nanpercentile(data.values, 3) / 10) * 10
                    vmax = np.ceil(np.nanpercentile(data.values, 97) / 10) * 10
                    month_str = datetime.strptime(month, "%m%y").strftime("%b %Y")
                    create_global_plot(ds_test[var], statistic_name=var_name, 
                        wavelength=wavelength, output_path=f'{folder}gb_ch{channel}/{file}', 
                        count_mask=ds_test.count_mask, satellite=sattlite.capitalize(), season=month_str, vmin=vmin, vmax=vmax)
            except:
                errors.append(f"Error processing {sattlite} {month} ch{channel}")